### 1. Environment Initialization and Configuration
Importing core dependencies for inference and establishing the configuration space. The computation device is explicitly bound to the CPU to ensure stability and compliance with standard inference environment constraints. We implement a dynamic path traversal mechanism (`find_path_safe`) to programmatically locate hidden test soundscapes and pre-trained model weights, thereby decoupling the logic from fragile, hardcoded directory structures.

import os
import gc
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from tqdm.auto import tqdm
import timm

import warnings
warnings.filterwarnings('ignore')

class CFG:
    device = torch.device("cpu") # Force CPU for Inference
    sr = 32000
    duration = 5 
    n_mels = 128
    fmin = 40
    fmax = 15000
    
    # PCEN Parameters - MUST MATCH TRAINING
    pcen_alpha = 0.98
    pcen_delta = 2.0
    pcen_r = 0.5
    pcen_s = 0.025 

# Helper to find paths automatically
def find_path_safe(name, is_dir=False):
    for root, dirs, files in os.walk('/kaggle/input'):
        targets = dirs if is_dir else files
        if name in targets:
            return Path(root) / name
    return None

# Find competition folders
BASE_DIR = find_path_safe("sample_submission.csv").parent
TEST_DIR = find_path_safe("test_soundscapes", is_dir=True)
SAMPLE_SUB = find_path_safe("sample_submission.csv")

# Find your uploaded model file
FINAL_WEIGHTS = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith(".pth"): # Picks up your uploaded weights
            FINAL_WEIGHTS = Path(root) / f
            break

print(f" Competition Directory: {BASE_DIR}")
print(f" Weights Found: {FINAL_WEIGHTS}")

### 2. Acoustic Feature Extraction and Model Instantiation
Defining the Per-Channel Energy Normalization (PCEN) pipeline. It is mathematically critical that the parameters $(\alpha, \delta, r, s)$ strictly mirror the training domain to prevent distribution shift. The `BirdConvNeXt` backbone is initialized with `pretrained=False` to strictly satisfy the competition's zero-internet execution mandate. The expert weights derived from our dual-phase training regime are subsequently mapped to the instantiated architecture.

def compute_pcen(audio_path, offset=0):
    """
    Constructs the PCEN spectrogram (Spectrum) for a 5s window.
    """
    try:
        # 1. Load 5 seconds of audio
        y, _ = librosa.load(audio_path, sr=CFG.sr, offset=offset, duration=CFG.duration)
        if len(y) < CFG.sr * CFG.duration:
            y = np.pad(y, (0, CFG.sr * CFG.duration - len(y)))

        # 2. Compute Mel Spectrogram
        S = librosa.feature.melspectrogram(
            y=y, sr=CFG.sr, n_mels=CFG.n_mels, 
            fmin=CFG.fmin, fmax=CFG.fmax, power=1
        )

        # 3. Apply PCEN (The background noise filter)
        spec = librosa.pcen(
            S * (2**31), sr=CFG.sr, gain=CFG.pcen_s, 
            bias=CFG.pcen_delta, power=CFG.pcen_r, b=CFG.pcen_alpha
        )

        # 4. Normalize to [0, 255]
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)
        return (spec * 255).astype(np.uint8)
    except Exception:
        return np.zeros((CFG.n_mels, 313), dtype=np.uint8)

class BirdConvNeXt(nn.Module):
    def __init__(self, model_name='convnext_tiny.fb_in1k', num_classes=234):
        super().__init__()
        # pretrained=False is required for no-internet submission
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=3)
        n_features = self.backbone.head.fc.in_features
        self.backbone.head.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(n_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

# Load Model
model = BirdConvNeXt(num_classes=234)
if FINAL_WEIGHTS:
    model.load_state_dict(torch.load(FINAL_WEIGHTS, map_location=CFG.device))
model.to(CFG.device)
model.eval()
print(" Model Architecture and Weights Loaded.")

### 3. Inference Pipeline and Submission Generation
Constructing the temporal sliding window mechanism. The pipeline parses contiguous 60-second test soundscapes into discrete, non-overlapping 5-second arrays ($t_{window} = 5s$). Each segment tensor is propagated forward through the network, outputting a 234-dimensional probability vector via a Sigmoid activation function $\sigma(z_i) = \frac{1}{1 + e^{-z_i}}$. The predictions are sequentially aggregated and formatted into the canonical matrix structure required for final submission.

# 1. Get official species columns
sample_sub = pd.read_csv(SAMPLE_SUB)
species_cols = list(sample_sub.columns[1:])

# 2. Get test files (Only exists during actual submission)
test_files = list(TEST_DIR.glob("*.ogg")) if TEST_DIR else []
submission_rows = []

if not test_files:
    print(" Hidden test files not found. Creating dummy for Save Version...")
    sample_sub.to_csv("submission.csv", index=False)
else:
    print(f" Processing {len(test_files)} forest soundscapes...")
    for file_path in tqdm(test_files):
        file_id = file_path.stem
        
        # Split 1-minute audio into 12 segments of 5s each
        for i in range(12):
            start_sec = i * 5
            row_id = f"{file_id}_{start_sec + 5}"
            
            # Create the Spectrum
            spec = compute_pcen(file_path, offset=start_sec)
            
            # CNNs expect 3 channels (RGB)
            image = np.stack([spec, spec, spec], axis=0)
            tensor = torch.tensor(image).float().unsqueeze(0).to(CFG.device) / 255.0
            
            # Predict
            with torch.no_grad():
                logits = model(tensor)
                probs = torch.sigmoid(logits).numpy()[0]
            
            # Prepare row for CSV
            row = {"row_id": row_id}
            row.update(dict(zip(species_cols, probs)))
            submission_rows.append(row)

    # 3. Finalize
    submission_df = pd.DataFrame(submission_rows)
    # Reorder columns to match competition requirements
    submission_df = submission_df[["row_id"] + species_cols]
    submission_df.to_csv("submission.csv", index=False)
    print(" submission.csv successfully generated!")